In [ ]:
# %pip install -U faster-whisper

# winget install Gyan.FFmpeg

# import shutil
# print("FFmpeg:", shutil.which("ffmpeg"))

In [ ]:
 !pip install git+https://github.com/openai/whisper.git
 !sudo apt update && sudo apt install ffmpeg

In [ ]:
### Configuration ###

from pathlib import Path

PROJECT_DIR = Path.cwd() / "transcribeWhisper"

VIDEO_DIR = PROJECT_DIR / "videos"
AUDIO_DIR = PROJECT_DIR / "sounds"
TRANSCRIPT_DIR = PROJECT_DIR / "transcribes"
EXTRACTED_AUDIO_DIR = AUDIO_DIR / "extracted"

# Whisper
MODEL_SIZE = "medium.en"
LANGUAGE = "en"


VIDEO_EXTENSIONS = {
    ".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v"}

AUDIO_EXTENSIONS = {
    ".mp3", ".wav", ".m4a", ".flac", ".ogg", ".wma"}

for folder in [VIDEO_DIR, AUDIO_DIR, TRANSCRIPT_DIR, EXTRACTED_AUDIO_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())

In [ ]:
### Model ###

from faster_whisper import WhisperModel

DEVICE = "cuda"  # cpu

COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE)

print(f"Loaded {MODEL_SIZE} on {DEVICE}")

In [ ]:
### Transcription Function ###

def transcribe_audio(audio_path, output_dir=TRANSCRIPT_DIR):
    audio_path = Path(audio_path)
    output_dir = Path(output_dir)

    if not audio_path.is_file():
        print(f"File not found: {audio_path}")
        return None

    output_dir.mkdir(parents=True, exist_ok=True)

    transcript_path = output_dir / f"{audio_path.stem}.txt"

    if transcript_path.exists():
        print(f"Already exists: {transcript_path.name}")
        return transcript_path

    segments, info = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        beam_size=5,
        vad_filter=True)

    text = " ".join(
        segment.text.strip()
        for segment in segments).strip()

    with transcript_path.open("w", encoding="utf-8") as f:
        f.write(text)

    print(f"Transcribed: {audio_path.name} "
        f"({info.language})")

    return transcript_path

In [ ]:
### VIDEO ###

### 1. Extract Audio
import subprocess
import shutil

def extract_audio(video_path, output_path):
    video_path = Path(video_path)
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True, exist_ok=True
    )

    command = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-b:a", "192k",
        str(output_path)
    ]

    subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True
    )

    return output_path

### 2. Workflow
video_files = sorted(
    p for p in VIDEO_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in VIDEO_EXTENSIONS
)

print(f"Found {len(video_files)} video files")

for video_path in video_files:
    try:
        audio_path = (
            EXTRACTED_AUDIO_DIR
            / f"{video_path.stem}.mp3"
        )

        # Extract audio only if it is not already present
        if not audio_path.exists():
            extract_audio(video_path, audio_path)

        # Save transcript using the video's filename
        transcribe_audio(
            audio_path,
            output_dir=TRANSCRIPT_DIR
        )

    except Exception as e:
        print(f"Error processing {video_path.name}: {e}")

In [ ]:
### AUDIO ###

audio_files = sorted(
    p for p in AUDIO_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in AUDIO_EXTENSIONS
)

print(f"Found {len(audio_files)} audio files")

for audio_path in audio_files:
    try:
        transcribe_audio(
            audio_path,
            output_dir=TRANSCRIPT_DIR
        )

    except Exception as e:
        print(f"Error processing {audio_path.name}: {e}")

### Creating audio files from videos

In [ ]:
# import necessary libraries
import moviepy.editor as mp
import os

video_folder = "videos" # Video folder location
sounds_folder = "sounds" # Sound folder location

video_files = os.listdir(video_folder) # list of video files
index = 0

while index < len(video_files):
    video_file = video_files[index]
    video_path = os.path.join(video_folder, video_file)

    if os.path.isfile(video_path) and video_path.endswith(".mp4"):
        video = mp.VideoFileClip(video_path)
        audio_filename = video_path.replace(".mp4", "").replace("videos\\", "")
        audio_path = os.path.join(sounds_folder, audio_filename)
        video.audio.write_audiofile(f"{audio_path}.mp3")
    index += 1

                                                                    
chunk:   1%|▏         | 23/1690 [07:12<8:42:08, 18.79s/it, now=None]

MoviePy - Writing audio in sounds\001-The Complete Focus Mastery Course Brain Concentration Promo Video-fMTi-git.ir.mp3












                                                                    
chunk:   1%|▏         | 23/1690 [07:13<8:43:13, 18.83s/it, now=None] 

MoviePy - Done.


*Transcibe the audio files*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
import whisper

model = whisper.load_model("medium.en")

sounds_folder = "/content/drive/MyDrive/Notebooks/transcribeWhisper/sounds/"
transcribe_folder = "/content/drive/MyDrive/Notebooks/transcribeWhisper/transcribes/"

for sounds in os.listdir(sounds_folder):
  if sounds.endswith(".mp3"):
    sound = os.path.join(sounds_folder, sounds)
    result = model.transcribe(sound)
    transcription_text = result["text"]

    transcription_filename = os.path.splitext(sounds)[0] + ".txt"
    transcription_path = os.path.join(transcribe_folder, transcription_filename)

    with open(transcription_path, 'w') as f:
      f.write(transcription_text)


/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
